# Fixed straight-line versus quadratic-Bezier comparison — one authorized execution

This notebook runs the single authorized 6-target × 3-seed × 2-condition comparison. It uses the public repository, exact runner commit `398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b`, and authorization commit `cc857407ed431c5583fd9e1c02a0ba619a8c187a`. Full outputs are written to Google Drive. No GPU is required because the painter uses Pillow and NumPy on the CPU.

Run code cells 1–6 in order. Do not interrupt the execution cell. If an `.incomplete` directory exists or the run fails, preserve it and stop—do not delete it or rerun. Cell 6 will reveal only whether blinded review is required. Do not run Cell 7 unless the blinded review has already been completed or Cell 6 says that no blinded review is required.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'quadratic-bezier-extension'
RUNNER_COMMIT = '398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b'
AUTHORIZATION_COMMIT = 'cc857407ed431c5583fd9e1c02a0ba619a8c187a'
AUTHORIZATION_REPO_PATH = 'configs/quadratic-bezier-execution-authorization-2026-09-04.json'
AUTHORIZATION_PATH = Path('/content/quadratic-bezier-execution-authorization-2026-09-04.json')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
clone = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    print(clone.stdout)
    print(clone.stderr)
    raise RuntimeError('Public repository clone failed.')
authorization_text = subprocess.check_output(
    ['git', 'show', f'{AUTHORIZATION_COMMIT}:{AUTHORIZATION_REPO_PATH}'],
    cwd=REPO_DIR,
    text=True,
)
AUTHORIZATION_PATH.write_text(authorization_text, encoding='utf-8')
authorization = json.loads(authorization_text)
subprocess.run(['git', 'checkout', '--quiet', '--detach', RUNNER_COMMIT], cwd=REPO_DIR, check=True)
observed_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert observed_commit == RUNNER_COMMIT, observed_commit
assert authorization['authorized_runner_commit'] == RUNNER_COMMIT
assert authorization['execution_authorized'] is True
assert authorization['completed_executions'] == 0
assert subprocess.check_output(['git', 'status', '--short'], cwd=REPO_DIR, text=True).strip() == ''
print('CELL 1 COMPLETE — EXACT RUNNER AND ONE-TIME AUTHORIZATION PINNED')
print('runner:', RUNNER_COMMIT)
print('authorization commit:', AUTHORIZATION_COMMIT)

In [ ]:
import subprocess
from pathlib import Path

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
test_run = subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
EXECUTION_PYTEST_LOG = Path('/content/quadratic_bezier_execution_pytest.txt')
EXECUTION_PYTEST_LOG.write_text(test_run.stdout + test_run.stderr, encoding='utf-8')
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0, 'The complete test suite failed; do not execute.'
print('CELL 2 COMPLETE — COMPLETE TEST SUITE PASSED')

In [ ]:
import json
import subprocess
import sys

sys.path.insert(0, str(REPO_DIR / 'src'))
from latent_stroke_dynamics.quadratic_bezier_comparison import (
    validate_execution_authorization,
    validate_only_comparison_report,
)

FREEZE_PATH = REPO_DIR / 'configs/quadratic-bezier-target-freeze-2026-09-04.json'
validation = validate_only_comparison_report(FREEZE_PATH)
validated_authorization = validate_execution_authorization(
    AUTHORIZATION_PATH,
    source_commit=RUNNER_COMMIT,
    freeze_path=FREEZE_PATH,
)
assert validation['status'] == 'quadratic_bezier_comparison_runner_valid_no_outputs'
assert validation['target_freeze_sha256'] == 'd7606da143f9b64e145cac95759dd3b29ff4a8bbf5edcf7aecce4d068930b494'
assert validation['target_set_sha256'] == '26bada941bfd8f49f09333d70d397364e82f5ddbb6e1228324f24fb9d2b30bfd'
assert validation['expected_run_count'] == 36
assert validation['output_side_effects'] is False
assert validated_authorization['maximum_completed_executions'] == 1
assert validated_authorization['expected_output_name'] == 'quadratic-bezier-fixed-comparison-v1'
print('CELL 3 COMPLETE — TARGET FREEZE, RUNNER, AND AUTHORIZATION VALIDATED')
print('environment:', validation['environment'])

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
OUTPUT_PARENT = Path('/content/drive/MyDrive/latent-stroke-dynamics-rgb')
OUTPUT_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1'
INCOMPLETE_DIR = OUTPUT_PARENT / 'quadratic-bezier-fixed-comparison-v1.incomplete'
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Completed output already exists; do not rerun: {OUTPUT_DIR}')
if INCOMPLETE_DIR.exists():
    raise FileExistsError(f'Preserve and report the incomplete output; do not delete or rerun: {INCOMPLETE_DIR}')
print('CELL 4 COMPLETE — DRIVE MOUNTED; FRESH FIXED OUTPUT PATH CONFIRMED')
print('output:', OUTPUT_DIR)

In [ ]:
import json
from pathlib import Path
import subprocess
import time

EXECUTION_LOG = Path('/content/quadratic_bezier_execution_log.txt')
command = [
    'python',
    'run_quadratic_bezier_comparison.py',
    '--execute',
    '--authorization',
    str(AUTHORIZATION_PATH),
    '--output-dir',
    str(OUTPUT_DIR),
    '--source-commit',
    RUNNER_COMMIT,
]
started = time.monotonic()
print('STARTING THE ONE FIXED 36-RUN COMPARISON — DO NOT INTERRUPT')
with EXECUTION_LOG.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
        while process.poll() is None:
            elapsed = time.monotonic() - started
            print(f'Comparison still running — elapsed {elapsed / 60:.1f} minutes')
            time.sleep(30)
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        print('Execution was interrupted. Preserve the .incomplete directory and do not rerun.')
        raise
return_code = process.returncode
if return_code != 0:
    tail = EXECUTION_LOG.read_text(encoding='utf-8')[-5000:]
    print(tail)
    raise RuntimeError('Fixed comparison failed. Preserve the .incomplete directory and stop.')
assert OUTPUT_DIR.is_dir()
assert not INCOMPLETE_DIR.exists()
print('CELL 5 COMPLETE — ONE FIXED COMPARISON FINISHED')
print(f'elapsed minutes: {(time.monotonic() - started) / 60:.2f}')

In [ ]:
from hashlib import sha256
import json
from pathlib import Path
from google.colab import files

SUMMARY_PATH = OUTPUT_DIR / 'aggregate_summary.json'
SUMMARY_SHA_PATH = OUTPUT_DIR / 'aggregate_summary.sha256'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
observed_summary_sha = sha256(SUMMARY_PATH.read_bytes()).hexdigest()
assert SUMMARY_SHA_PATH.read_text(encoding='utf-8').strip() == observed_summary_sha
assert summary['status'] == 'quadratic_bezier_fixed_comparison_complete'
assert summary['completed_run_count'] == 36
assert summary['completed_pair_count'] == 18
assert summary['integrity_passed'] is True
assert summary['training_performed'] is False
assert summary['learned_model_used'] is False
assert summary['closed_experiments_changed'] is False
verified_artifacts = 0
for relative_path, expected_sha in summary['artifact_sha256'].items():
    artifact = OUTPUT_DIR / relative_path
    assert artifact.is_file(), relative_path
    assert sha256(artifact.read_bytes()).hexdigest() == expected_sha, relative_path
    verified_artifacts += 1
qualitative_required = bool(summary['quantitative_decision']['qualitative_review_required'])
blind_handoff = {
    'status': 'quadratic_bezier_blind_handoff_ready',
    'source_commit': RUNNER_COMMIT,
    'authorization_commit': AUTHORIZATION_COMMIT,
    'aggregate_summary_sha256': observed_summary_sha,
    'target_set_sha256': summary['target_set_sha256'],
    'completed_run_count': summary['completed_run_count'],
    'completed_pair_count': summary['completed_pair_count'],
    'integrity_passed': summary['integrity_passed'],
    'verified_artifact_count': verified_artifacts,
    'qualitative_review_required': qualitative_required,
    'quantitative_values_withheld_for_blinding': qualitative_required,
}
BLIND_HANDOFF_PATH = Path('/content/quadratic_bezier_blind_handoff.json')
BLIND_HANDOFF_PATH.write_text(json.dumps(blind_handoff, indent=2, sort_keys=True) + '\n', encoding='utf-8')
files.download(str(BLIND_HANDOFF_PATH))
if qualitative_required:
    files.download(str(OUTPUT_DIR / 'blinded_review_montage.png'))
    files.download(str(OUTPUT_DIR / 'blinded_review_sheet.csv'))
    print('CELL 6 COMPLETE — BLINDED REVIEW IS REQUIRED')
    print('Return only the blind handoff JSON, blinded montage, and blank review sheet.')
    print('Do not open the mapping, aggregate summary, numerical plots, or execution log yet.')
else:
    for name in [
        'aggregate_summary.json',
        'aggregate_summary.sha256',
        'quantitative_decision.json',
        'target_seed_pair_metrics.csv',
        'mean_512_mse_by_primitive.png',
        'per_target_curve_ratio.png',
        'aggregate_progress_by_primitive.png',
    ]:
        files.download(str(OUTPUT_DIR / name))
    files.download(str(EXECUTION_PYTEST_LOG))
    files.download(str(EXECUTION_LOG))
    print('CELL 6 COMPLETE — NO BLINDED REVIEW GATE REQUIRED; NUMERICAL HANDOFF DOWNLOADED')
print('verified artifacts:', verified_artifacts)

## Cell 7 — reveal only after blinded review

If Cell 6 says blinded review is required, stop and return the three blind-review files first. After that review is recorded, set the flag below to `True` and run only Cell 7. If no blinded review is required, Cell 6 already downloaded the numerical handoff and this cell is unnecessary.

In [ ]:
from google.colab import files

REVEAL_AFTER_BLINDED_REVIEW = False
if qualitative_required:
    if not REVEAL_AFTER_BLINDED_REVIEW:
        raise RuntimeError('STOP: complete and record the blinded review before revealing method identities or metrics.')
    reveal_names = [
        'aggregate_summary.json',
        'aggregate_summary.sha256',
        'quantitative_decision.json',
        'target_seed_pair_metrics.csv',
        'mean_512_mse_by_primitive.png',
        'per_target_curve_ratio.png',
        'aggregate_progress_by_primitive.png',
        'blinded_mapping_do_not_open_before_review.json',
    ]
    for name in reveal_names:
        files.download(str(OUTPUT_DIR / name))
    files.download(str(EXECUTION_PYTEST_LOG))
    files.download(str(EXECUTION_LOG))
    print('CELL 7 COMPLETE — POST-REVIEW NUMERICAL AND IDENTITY HANDOFF DOWNLOADED')
else:
    print('Cell 7 not needed: Cell 6 already downloaded the numerical handoff.')